# Mileforum — Aprendiz Ajuste Bimestral (Policy Fine-Tune)
**Versión:** v0.1 (generada 2026-02-18)

Este notebook **no construye** modelos (no es “fábrica”). Su única función es:

1) Cargar el **bundle multicefálico** existente (zip)  
2) Cargar el **histórico bimestral** generado por el notebook día-a-día (*Aprendiz Episódico*)  
3) Ajustar **solo** la capa **policy** (o, si tu modelo no expone policy entrenable, entrenar un `PolicyAdapter` prudencial)  
4) Consolidar y versionar:
   - `allowed_actions_by_phase.json` (allowlist operacional)
   - `actions_added_this_bimester.json` (diff)
   - `policy_adapter.pt` / `policy_only_state_dict.pt`
   - `bimester_report.json`

> **Bundle esperado:** `domain_multiceph_bundle_v1.zip` (ponlo en el mismo folder que este notebook, o ajusta `BUNDLE_ZIP_PATH`).


In [ ]:
# =========================
# Celda 1 — Configuración
# =========================
import os, json, re, math, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

# --- Paths (ajusta si hace falta) ---
BUNDLE_ZIP_PATH = "domain_multiceph_bundle_v1.zip"   # <- tu zip multicefálico
WORKDIR = Path("./_bimestral_work").resolve()
BUNDLE_DIR = WORKDIR / "bundle_extracted"
OUTPUT_DIR = Path("./bimester_outputs").resolve()

# Histórico del día-a-día (Aprendiz Episódico)
LEARNING_LOG_PATH = Path("./local_store/learning_log.jsonl")  # por default en tu diario
ACTION_DICT_PATH  = Path("./local_store/action_dictionary_state.json")
SOFT_DICT_PATH    = Path("./local_store/soft_dictionary_state.json")

# Ventana bimestral (ISO 8601: YYYY-MM-DD)
BIMESTER_START = "2026-01-01"
BIMESTER_END   = "2026-02-29"

# Alcance de ajuste
# - "per_domain" recomendado (una policy por dominio)
# - "global" si tu policy es shared
TARGET_SCOPE = "per_domain"

# Dominio objetivo (si None, el notebook lista dominios y tú eliges)
TARGET_DOMAIN: Optional[str] = None

# --- Entrenamiento policy ---
SEED = 7
EPOCHS = 8
BATCH_SIZE = 64
LR = 2e-4
WEIGHT_DECAY = 1e-4

# Ponderación de señal humana (prudencial)
W_CORRECTION = 2.0   # eventos donde el profesional NO aceptó sugerencia (gold)
W_CONFIRM    = 1.0   # aceptó sugerencia (silver)

# Prudencia: bajar el peso en baja claridad (por default)
W_LOW_CLARITY = 0.5  # multiplica el peso si clarity.ok == False

# Allowlist policy:
# - "catalog_fast": allowlist = catálogo actualizado completo
# - "conservative": allowlist solo si una acción nueva aparece >= MIN_NEW_ACTION_FREQ
ALLOWLIST_POLICY = "conservative"
MIN_NEW_ACTION_FREQ = 2

# Guardado
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_OUT = OUTPUT_DIR / f"bimester_{RUN_ID}"
RUN_OUT.mkdir(parents=True, exist_ok=True)

print("RUN_OUT =", RUN_OUT)


In [ ]:
# =========================
# Celda 2 — Dependencias
# =========================
import random
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("Este notebook requiere PyTorch. Instálalo en tu entorno antes de correr.") from e

def seed_all(seed:int=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


In [ ]:
# ===========================================
# Celda 3 — Utilidades: JSONL + timestamps
# ===========================================
from datetime import datetime

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def parse_ts(ts: str) -> datetime:
    # Formato esperado: "%Y-%m-%dT%H:%M:%S%z" (como en el diario)
    # Fallback: intenta ISO
    try:
        return datetime.strptime(ts, "%Y-%m-%dT%H:%M:%S%z")
    except Exception:
        return datetime.fromisoformat(ts.replace("Z","+00:00"))

def iter_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def within_window(e: Dict[str,Any], start: str, end: str) -> bool:
    t = parse_ts(e["timestamp"])
    # ancla naive a tz de t
    s = datetime.fromisoformat(start).replace(tzinfo=t.tzinfo)
    ed = datetime.fromisoformat(end).replace(tzinfo=t.tzinfo)
    return (t >= s) and (t <= ed)

def safe_get(d: Dict[str,Any], path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


In [ ]:
# ===========================================
# Celda 4 — Extraer bundle multicefálico (zip)
# ===========================================
import zipfile

WORKDIR.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

zip_path = Path(BUNDLE_ZIP_PATH)
if not zip_path.exists():
    raise FileNotFoundError(
        f"No encontré el zip: {zip_path.resolve()}\n"
        "Pon `domain_multiceph_bundle_v1.zip` junto a este notebook o ajusta BUNDLE_ZIP_PATH."
    )

bundle_hash = sha256_file(zip_path)
print("Bundle SHA256:", bundle_hash)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(BUNDLE_DIR)

# Heurística: listar dominios disponibles
domains = []
for p in BUNDLE_DIR.rglob("*"):
    if p.is_dir() and ((p / "domain.json").exists() or (p / "config.json").exists()):
        domains.append(p)

if not domains:
    domains = [p for p in BUNDLE_DIR.iterdir() if p.is_dir()]

print("Candidatos de dominio (paths):")
for i, d in enumerate(domains[:100]):
    print(f"  [{i}] {d.relative_to(BUNDLE_DIR)}")

print("\nTIP: pon TARGET_DOMAIN con el nombre exacto (p.ej. 'veterinaria', 'logistica', etc.)")


In [ ]:
# ==========================================================
# Celda 5 — Resolver dominio objetivo + paths del modelo base
# ==========================================================
def pick_domain(domains_paths: List[Path], target_domain: Optional[str]) -> Path:
    if not domains_paths:
        raise RuntimeError("No encontré subcarpetas de dominio dentro del bundle.")
    if target_domain is None:
        return domains_paths[0]
    td = target_domain.lower()
    for p in domains_paths:
        if td in str(p).lower():
            return p
    raise ValueError(f"No pude resolver TARGET_DOMAIN='{target_domain}'. Revisa la lista impresa.")

DOMAIN_DIR = pick_domain(domains, TARGET_DOMAIN)
print("DOMAIN_DIR:", DOMAIN_DIR)

model_candidates = sorted(list(DOMAIN_DIR.rglob("*.pt")) + list(DOMAIN_DIR.rglob("*.pth")))
json_candidates = sorted(list(DOMAIN_DIR.rglob("*.json")))

print("\nModel candidates:")
for p in model_candidates[:30]:
    print(" -", p.relative_to(DOMAIN_DIR))

print("\nMetadata candidates:")
for p in json_candidates[:30]:
    print(" -", p.relative_to(DOMAIN_DIR))

MODEL_PATH = model_candidates[0] if model_candidates else None
if MODEL_PATH is None:
    raise FileNotFoundError("No encontré archivos .pt/.pth en el dominio. Ajusta el bundle o la heurística.")
print("\nMODEL_PATH:", MODEL_PATH)


In [ ]:
# ======================================================
# Celda 6 — Cargar histórico bimestral (Aprendiz Episódico)
# ======================================================
if not LEARNING_LOG_PATH.exists():
    raise FileNotFoundError(
        f"No encontré learning log en: {LEARNING_LOG_PATH.resolve()}\n"
        "Copia tu carpeta local_store/ junto a este notebook (o ajusta LEARNING_LOG_PATH)."
    )

events = [e for e in iter_jsonl(LEARNING_LOG_PATH) if within_window(e, BIMESTER_START, BIMESTER_END)]
print("Eventos en ventana:", len(events))

if TARGET_SCOPE == "per_domain":
    if TARGET_DOMAIN is None:
        TARGET_DOMAIN = DOMAIN_DIR.name
        print("Inferí TARGET_DOMAIN =", TARGET_DOMAIN)
    events = [e for e in events if e.get("domain") == TARGET_DOMAIN]
    print("Eventos (dominio filtrado):", len(events))

action_state = json.loads(ACTION_DICT_PATH.read_text(encoding="utf-8")) if ACTION_DICT_PATH.exists() else {"actions_by_phase": {}}
soft_state   = json.loads(SOFT_DICT_PATH.read_text(encoding="utf-8")) if SOFT_DICT_PATH.exists() else {"custom_soft_vars": []}

actions_by_phase = action_state.get("actions_by_phase", {})
print("Fases en action_dictionary:", list(actions_by_phase.keys())[:30])
print("Custom soft vars:", len(soft_state.get("custom_soft_vars", [])))


In [ ]:
# ======================================================
# Celda 7 — Extraer samples (state, target, weights)
# ======================================================
from collections import Counter

def extract_sample(e: Dict[str,Any]) -> Optional[Dict[str,Any]]:
    bi = e.get("backbone_inference") or {}
    ae = e.get("action_execution") or None
    if ae is None:
        return None

    phase = bi.get("phase", "unknown")
    probs = bi.get("phase_probs", {}) or {}
    clarity = bi.get("clarity", {}) or {}
    R = bi.get("R_score", None)

    soft_tags = safe_get(e, ["soft_context","soft_tags"], []) or []
    accepted = ae.get("user_accepted_suggestion", False)

    w = W_CORRECTION if (accepted is False) else W_CONFIRM
    if clarity.get("ok") is False:
        w *= W_LOW_CLARITY

    return {
        "domain": e.get("domain"),
        "node_id": e.get("node_id"),
        "phase": phase,
        "phase_probs": probs,
        "clarity": clarity,
        "R_score": float(R) if R is not None else 0.0,
        "soft_tags": soft_tags,
        "target_action_id": ae.get("action_id"),
        "target_action_label": ae.get("action_label"),
        "accepted": bool(accepted),
        "action_known": bool(ae.get("action_known", True)),
        "weight": float(w),
        "timestamp": e.get("timestamp"),
        "has_learning_event": ("learning_event" in e),
    }

samples = [s for e in events if (s := extract_sample(e)) is not None and s["target_action_id"]]
print("Samples supervisados:", len(samples))

new_actions_obs = [s for s in samples if s["action_known"] is False]
print("Acciones nuevas observadas:", len(new_actions_obs))

freq_new = Counter([ (s["phase"], s["target_action_id"]) for s in new_actions_obs ])
print("Top nuevas (phase, action_id) por frecuencia:")
for (ph, aid), c in freq_new.most_common(10):
    print(" ", ph, aid, c)


In [ ]:
# ======================================================
# Celda 8 — Build catálogo + allowlist (by phase)
# ======================================================
def normalize_label(x: Optional[str]) -> str:
    if not x:
        return ""
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

catalog = {ph: list(v) for ph, v in actions_by_phase.items()}

added = []
for s in new_actions_obs:
    ph = s["phase"]
    action_id = s["target_action_id"]
    label = s["target_action_label"] or action_id
    item = {"action_id": action_id, "label": label}
    catalog.setdefault(ph, [])

    existing_ids = {a.get("action_id") for a in catalog[ph] if isinstance(a, dict)}
    if action_id not in existing_ids:
        catalog[ph].append(item)
        added.append({"phase": ph, **item, "first_seen": s["timestamp"], "domain": s["domain"], "node_id": s["node_id"]})

allowed = {ph: [] for ph in catalog.keys()}
for ph, lst in catalog.items():
    if ALLOWLIST_POLICY == "catalog_fast":
        allowed[ph] = lst
    else:
        allow_ids = set()
        prior_ids = {a.get("action_id") for a in actions_by_phase.get(ph, []) if isinstance(a, dict)}
        allow_ids |= prior_ids
        for a in lst:
            aid = a.get("action_id") if isinstance(a, dict) else None
            if aid and freq_new.get((ph, aid), 0) >= MIN_NEW_ACTION_FREQ:
                allow_ids.add(aid)
        allowed[ph] = [a for a in lst if isinstance(a, dict) and a.get("action_id") in allow_ids]

print("Acciones agregadas al catálogo:", len(added))
print("Allowlist sizes:")
for ph in sorted(allowed.keys()):
    print(f" - {ph}: {len(allowed[ph])} allowed / {len(catalog[ph])} catalog")

(RUN_OUT / "actions_added_this_bimester.json").write_text(json.dumps(added, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_OUT / "allowed_actions_by_phase.json").write_text(json.dumps(allowed, ensure_ascii=False, indent=2), encoding="utf-8")


In [ ]:
# ======================================================
# Celda 9 — Construcción de features (evento -> vector)
# ======================================================
import numpy as np

phase_prob_keys = sorted({k for s in samples for k in (s["phase_probs"] or {}).keys()})
print("phase_prob_keys:", phase_prob_keys)

soft_vocab = sorted({t for s in samples for t in (s["soft_tags"] or [])})
soft_index = {t:i for i,t in enumerate(soft_vocab)}
print("soft_vocab size:", len(soft_vocab))

def vectorize(s: Dict[str,Any]) -> np.ndarray:
    probs = s["phase_probs"] or {}
    clarity = s["clarity"] or {}

    v = []
    v.extend([float(probs.get(k, 0.0)) for k in phase_prob_keys])

    v.append(1.0 if clarity.get("ok") else 0.0)
    v.append(float(clarity.get("pmax", 0.0)))
    v.append(float(clarity.get("gap", 0.0)))
    v.append(float(clarity.get("entropy", 0.0)))

    v.append(float(s.get("R_score", 0.0)))

    st = np.zeros(len(soft_vocab), dtype=np.float32)
    for t in (s.get("soft_tags") or []):
        if t in soft_index:
            st[soft_index[t]] = 1.0
    v.extend(st.tolist())
    return np.array(v, dtype=np.float32)

X = np.stack([vectorize(s) for s in samples], axis=0)
w = np.array([s["weight"] for s in samples], dtype=np.float32)

print("X shape:", X.shape)


In [ ]:
# ======================================================
# Celda 10 — Mapear acciones a clases (por allowlist)
# ======================================================
allow_action_ids = sorted({a["action_id"] for ph in allowed for a in allowed[ph]})
action_to_idx = {aid:i for i,aid in enumerate(allow_action_ids)}
idx_to_action = {i:aid for aid,i in action_to_idx.items()}
print("Allow action vocab size:", len(allow_action_ids))

keep = [i for i,s in enumerate(samples) if s["target_action_id"] in action_to_idx]
X2 = X[keep]
w2 = w[keep]
y2 = np.array([action_to_idx[samples[i]["target_action_id"]] for i in keep], dtype=np.int64)
print("Samples after allowlist filter:", len(y2))

N = len(y2)
idx = np.arange(N)
np.random.shuffle(idx)
cut = int(0.85 * N)
tr, va = idx[:cut], idx[cut:]

print("Train:", len(tr), "Val:", len(va))


In [ ]:
# ======================================================
# Celda 11 — Entrenar PolicyAdapter (robusto multi-dominio)
# ======================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class PolicyAdapter(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden: int = 256, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

in_dim = int(X2.shape[1])
out_dim = int(len(allow_action_ids))
adapter = PolicyAdapter(in_dim, out_dim).to(DEVICE)

def batch_iter(X, y, w, batch_size=64, shuffle=True):
    idx = np.arange(len(y))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(y), batch_size):
        b = idx[i:i+batch_size]
        yield X[b], y[b], w[b]

opt = torch.optim.AdamW(adapter.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def eval_split(X, y, w):
    adapter.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for xb, yb, wb in batch_iter(X, y, w, batch_size=512, shuffle=False):
            xb = torch.tensor(xb, device=DEVICE)
            yb = torch.tensor(yb, device=DEVICE)
            wb = torch.tensor(wb, device=DEVICE)
            logits = adapter(xb)
            loss = F.cross_entropy(logits, yb, reduction="none")
            loss = (loss * wb).mean()
            loss_sum += float(loss.item()) * len(yb)
            pred = logits.argmax(dim=-1)
            correct += int((pred == yb).sum().item())
            total += int(len(yb))
    return {"loss": loss_sum / max(total,1), "acc": correct / max(total,1)}

best = {"val_loss": float("inf"), "epoch": -1}
history = []

for ep in range(1, EPOCHS+1):
    adapter.train()
    for xb, yb, wb in batch_iter(X2[tr], y2[tr], w2[tr], batch_size=BATCH_SIZE, shuffle=True):
        xb = torch.tensor(xb, device=DEVICE)
        yb = torch.tensor(yb, device=DEVICE)
        wb = torch.tensor(wb, device=DEVICE)
        logits = adapter(xb)
        loss = F.cross_entropy(logits, yb, reduction="none")
        loss = (loss * wb).mean()
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        opt.step()

    tr_m = eval_split(X2[tr], y2[tr], w2[tr])
    va_m = eval_split(X2[va], y2[va], w2[va])
    history.append({"epoch": ep, "train": tr_m, "val": va_m})
    print(f"Epoch {ep:02d} | train loss {tr_m['loss']:.4f} acc {tr_m['acc']:.3f} | val loss {va_m['loss']:.4f} acc {va_m['acc']:.3f}")

    if va_m["loss"] < best["val_loss"]:
        best = {"val_loss": va_m["loss"], "epoch": ep}
        torch.save(adapter.state_dict(), RUN_OUT / "policy_adapter.pt")

print("Best:", best)
(RUN_OUT / "train_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")


In [ ]:
# ======================================================
# Celda 12 — Reporte + metadatos + export zip
# ======================================================
report = {
    "run_id": RUN_ID,
    "bundle_zip": str(Path(BUNDLE_ZIP_PATH).name),
    "bundle_sha256": bundle_hash,
    "domain_dir": str(DOMAIN_DIR.relative_to(BUNDLE_DIR)),
    "target_domain": TARGET_DOMAIN,
    "time_window": {"start": BIMESTER_START, "end": BIMESTER_END},
    "counts": {
        "events_in_window": len(events),
        "samples_supervised": len(samples),
        "samples_after_allowlist": int(len(y2)),
        "new_actions_observed": int(len(new_actions_obs)),
        "actions_added_to_catalog": int(len(added)),
        "allow_vocab_size": int(len(allow_action_ids)),
    },
    "allowlist_policy": {
        "mode": ALLOWLIST_POLICY,
        "min_new_action_freq": MIN_NEW_ACTION_FREQ,
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "best_epoch": best["epoch"],
        "best_val_loss": best["val_loss"],
    },
    "feature_spec": {
        "phase_prob_keys": phase_prob_keys,
        "soft_vocab_size": len(soft_vocab),
        "in_dim": int(in_dim),
        "out_dim": int(out_dim),
    },
}

(RUN_OUT / "bimester_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_OUT / "soft_vocab.json").write_text(json.dumps(soft_vocab, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_OUT / "action_vocab.json").write_text(json.dumps(allow_action_ids, ensure_ascii=False, indent=2), encoding="utf-8")

print("Escritos en:", RUN_OUT)

import shutil
zip_out = shutil.make_archive(str(RUN_OUT), "zip", root_dir=RUN_OUT)
print("ZIP:", zip_out)
